# Results for model: ministral-3:8b

In [ ]:
import polars as pl
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np
from tqdm import tqdm

def load_and_preprocess_data():
    df = pl.read_parquet('./jane-street-real-time-market-data-forecasting/train.parquet')

    # Convert to numpy arrays for easier manipulation
    date_ids = df['date_id'].to_numpy()
    feature_00 = df['feature_00'].to_numpy()
    responder_6 = df['responder_6'].to_numpy()

    # Create a global TOP_QUANTILE feature
    top_quantile = []
    for date_id in tqdm(np.unique(date_ids), desc="Calculating TOP_QUANTILE"):
        mask = date_ids == date_id
        current_feature_00 = feature_00[mask]
        current_responder_6 = responder_6[mask]

        # Calculate quantile for each responder_6 group
        quantiles = np.percentile(current_feature_00, [85], interpolation='linear')[0]
        top_quantile.extend([quantiles] * np.sum(mask))

    # Add TOP_QUANTILE as a new feature
    df = df.with_columns(pl.Series("top_quantile", top_quantile))

    return df

def train_xgboost(df):
    # Prepare data
    X = df.drop(['date_id', 'responder_6', 'feature_00']).to_numpy()
    y = df['responder_6'].to_numpy()

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Train XGBoost model
    model = XGBRegressor(
        objective='reg:squarederror',
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    # Evaluate model
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    print(f"Test RMSE: {rmse:.4f}")

    return model

def main():
    df = load_and_preprocess_data()
    model = train_xgboost(df)

if __name__ == "__main__":
    main()